In [1]:

# ============================================================
# JEEVAN LINK - URGENCY MODEL TESTING SYSTEM
# ============================================================

import pandas as pd
import numpy as np
import joblib

from pathlib import Path

# ============================================================
# 1. LOAD TRAINED MODEL
# ============================================================

PROJECT_DIR = Path(r"C:\Users\ishit\jeevan_link_ml")

MODEL_DIR = PROJECT_DIR / "Models" / "Urgency_Model"

MODEL_FILE = MODEL_DIR / "urgency_model.pkl"

FEATURE_FILE = MODEL_DIR / "urgency_feature_names.pkl"

if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_FILE}")

if not FEATURE_FILE.exists():
    raise FileNotFoundError(f"Feature file not found: {FEATURE_FILE}")

model = joblib.load(MODEL_FILE)

feature_names = joblib.load(FEATURE_FILE)

print("Urgency model loaded successfully!")
print("Number of features:", len(feature_names))
print("Classes:", model.classes_)

# ============================================================
# 2. CREATE EMERGENCY INPUT FUNCTION
# ============================================================

def get_emergency_input():

    print("\nENTER EMERGENCY DETAILS")
    print("=" * 50)

    people_affected = int(
        input("Number of people affected: ")
    )

    people_trapped = int(
        input("Number of people trapped: ")
    )

    injuries_reported = int(
        input("Number of injuries reported: ")
    )

    medical_emergency = int(
        input("Medical emergency? (1=Yes, 0=No): ")
    )

    rescue_required = int(
        input("Rescue required? (1=Yes, 0=No): ")
    )

    food_required = int(
        input("Food required? (1=Yes, 0=No): ")
    )

    water_required = int(
        input("Water required? (1=Yes, 0=No): ")
    )

    shelter_required = int(
        input("Shelter required? (1=Yes, 0=No): ")
    )

    print("\nWater Level:")
    print("1. Stable")
    print("2. Rising")
    print("3. Rapidly Rising")

    water_choice = int(input("Choose option: "))

    water_level = {
        1: "Stable",
        2: "Rising",
        3: "Rapidly Rising"
    }

    if water_choice not in water_level:
        raise ValueError("Invalid water level choice.")

    print("\nBuilding Condition:")
    print("1. Intact")
    print("2. Damaged")
    print("3. Collapsing")
    print("4. Unknown")

    building_choice = int(input("Choose option: "))

    building_condition = {
        1: "Intact",
        2: "Damaged",
        3: "Collapsing",
        4: "Unknown"
    }

    if building_choice not in building_condition:
        raise ValueError("Invalid building condition.")

    print("\nLocation Risk:")
    print("1. Low")
    print("2. Medium")
    print("3. High")

    risk_choice = int(input("Choose option: "))

    location_risk = {
        1: "Low",
        2: "Medium",
        3: "High"
    }

    if risk_choice not in location_risk:
        raise ValueError("Invalid location risk.")

    emergency = {
        "people_affected": people_affected,
        "people_trapped": people_trapped,
        "injuries_reported": injuries_reported,
        "medical_emergency": medical_emergency,
        "rescue_required": rescue_required,
        "food_required": food_required,
        "water_required": water_required,
        "shelter_required": shelter_required,
        "water_level": water_level[water_choice],
        "building_condition": building_condition[building_choice],
        "location_risk": location_risk[risk_choice]
    }

    return emergency

# ============================================================
# 3. CONVERT INPUT INTO MODEL FEATURES
# ============================================================

def prepare_features(emergency):

    data = {
        "people_affected": emergency["people_affected"],
        "people_trapped": emergency["people_trapped"],
        "injuries_reported": emergency["injuries_reported"],
        "medical_emergency": emergency["medical_emergency"],
        "rescue_required": emergency["rescue_required"],
        "food_required": emergency["food_required"],
        "water_required": emergency["water_required"],
        "shelter_required": emergency["shelter_required"]
    }

    water_levels = [
        "Rapidly_Rising",
        "Rising",
        "Stable"
    ]

    for level in water_levels:
        data[f"water_level_{level}"] = int(
            emergency["water_level"].replace(" ", "_") == level
        )

    building_conditions = [
        "Collapsing",
        "Damaged",
        "Intact",
        "Unknown"
    ]

    for condition in building_conditions:
        data[f"building_condition_{condition}"] = int(
            emergency["building_condition"] == condition
        )

    location_risks = ["High", "Low", "Medium"]

    for risk in location_risks:
        data[f"location_risk_{risk}"] = int(
            emergency["location_risk"] == risk
        )

    input_df = pd.DataFrame([data])

    # Match the exact training feature order
    input_df = input_df.reindex(
        columns=feature_names,
        fill_value=0
    )

    return input_df.astype(float)

# ============================================================
# 4. PREDICT URGENCY
# ============================================================

def predict_urgency(emergency):

    input_df = prepare_features(emergency)

    prediction = model.predict(input_df)[0]

    probabilities = model.predict_proba(input_df)[0]

    probability_df = pd.DataFrame({
        "Urgency": model.classes_,
        "Probability": probabilities
    })

    probability_df["Probability"] = (
        probability_df["Probability"] * 100
    ).round(2)

    print("\n" + "=" * 55)
    print("JEEVAN LINK URGENCY PREDICTION")
    print("=" * 55)

    print("\nPredicted Urgency:", prediction)

    print("\nPrediction Probabilities:")
    print(probability_df.to_string(index=False))

    return prediction, probability_df

# ============================================================
# 5. TEST ONE EMERGENCY
# ============================================================

emergency = get_emergency_input()

prediction, probabilities = predict_urgency(emergency)

# ============================================================
# 6. TEST PREDEFINED SCENARIOS
# ============================================================

test_scenarios = {

    "Critical Scenario": {
        "people_affected": 100,
        "people_trapped": 20,
        "injuries_reported": 15,
        "medical_emergency": 1,
        "rescue_required": 1,
        "food_required": 1,
        "water_required": 1,
        "shelter_required": 1,
        "water_level": "Rapidly Rising",
        "building_condition": "Collapsing",
        "location_risk": "High"
    },

    "High Scenario": {
        "people_affected": 50,
        "people_trapped": 5,
        "injuries_reported": 3,
        "medical_emergency": 0,
        "rescue_required": 1,
        "food_required": 1,
        "water_required": 1,
        "shelter_required": 1,
        "water_level": "Rising",
        "building_condition": "Damaged",
        "location_risk": "High"
    },

    "Medium Scenario": {
        "people_affected": 20,
        "people_trapped": 0,
        "injuries_reported": 0,
        "medical_emergency": 0,
        "rescue_required": 0,
        "food_required": 1,
        "water_required": 1,
        "shelter_required": 1,
        "water_level": "Stable",
        "building_condition": "Damaged",
        "location_risk": "Medium"
    },

    "Low Scenario": {
        "people_affected": 2,
        "people_trapped": 0,
        "injuries_reported": 0,
        "medical_emergency": 0,
        "rescue_required": 0,
        "food_required": 0,
        "water_required": 0,
        "shelter_required": 0,
        "water_level": "Stable",
        "building_condition": "Intact",
        "location_risk": "Low"
    }
}

# ============================================================
# 7. RUN ALL SCENARIOS
# ============================================================

results = []

print("\n" + "=" * 65)
print("PREDEFINED EMERGENCY SCENARIO TESTING")
print("=" * 65)

for scenario_name, scenario in test_scenarios.items():

    prediction, probability_df = predict_urgency(scenario)

    results.append({
        "Scenario": scenario_name,
        "Predicted_Urgency": prediction,
        "Highest_Probability": probability_df[
            "Probability"
        ].max()
    })

results_df = pd.DataFrame(results)

print("\nFINAL SCENARIO RESULTS")
print(results_df.to_string(index=False))

# ============================================================
# 8. SAVE SCENARIO RESULTS
# ============================================================

results_df.to_csv(
    MODEL_DIR / "urgency_scenario_test_results.csv",
    index=False
)

print("\nScenario results saved successfully!")

print("\nUrgency model testing completed.")


Urgency model loaded successfully!
Number of features: 18
Classes: ['Critical' 'High' 'Low' 'Medium']

ENTER EMERGENCY DETAILS


Number of people affected:  100
Number of people trapped:  20
Number of injuries reported:  15
Medical emergency? (1=Yes, 0=No):  1
Rescue required? (1=Yes, 0=No):  1
Food required? (1=Yes, 0=No):  1
Water required? (1=Yes, 0=No):  1
Shelter required? (1=Yes, 0=No):  1



Water Level:
1. Stable
2. Rising
3. Rapidly Rising


Choose option:  3



Building Condition:
1. Intact
2. Damaged
3. Collapsing
4. Unknown


Choose option:  3



Location Risk:
1. Low
2. Medium
3. High


Choose option:  3



JEEVAN LINK URGENCY PREDICTION

Predicted Urgency: Critical

Prediction Probabilities:
 Urgency  Probability
Critical        98.19
    High         1.81
     Low         0.00
  Medium         0.00

PREDEFINED EMERGENCY SCENARIO TESTING

JEEVAN LINK URGENCY PREDICTION

Predicted Urgency: Critical

Prediction Probabilities:
 Urgency  Probability
Critical        98.19
    High         1.81
     Low         0.00
  Medium         0.00

JEEVAN LINK URGENCY PREDICTION

Predicted Urgency: High

Prediction Probabilities:
 Urgency  Probability
Critical         11.2
    High         88.8
     Low          0.0
  Medium          0.0

JEEVAN LINK URGENCY PREDICTION

Predicted Urgency: Medium

Prediction Probabilities:
 Urgency  Probability
Critical         0.03
    High         0.85
     Low         0.00
  Medium        99.12

JEEVAN LINK URGENCY PREDICTION

Predicted Urgency: Low

Prediction Probabilities:
 Urgency  Probability
Critical         0.03
    High         0.04
     Low        93.78
  Me

In [ ]:
#here via putting value accorddingly in output window we can see our modele is working completely fine 